# H2O Forward Equivariance Check

这个 notebook 使用单帧 `h2o_react.extxyz` 和单帧 `h2o_product.extxyz` 构建一个 joint graph，直接检查 EGNN / LEFTNet 前向传播后的输出等变性。

检查目标：

- `h_out` 对旋转不变
- `pos_out` 对旋转等变

In [1]:
from pathlib import Path
import sys

import torch
from e3nn import o3
from torch.utils.data import DataLoader

repo_root = Path.cwd()
if not (repo_root / 'dataset').exists() and (repo_root.parent / 'dataset').exists():
    repo_root = repo_root.parent
print(repo_root)

if str(repo_root.parent) not in sys.path:
    sys.path.insert(0, str(repo_root.parent))

from akmcgc.dataset import ReactionPairDataset
from akmcgc.model import EGNN, LEFTNet

torch.manual_seed(2024)
data_dir = repo_root / 'tests' / 'data'


/Users/wx/Desktop/yyxwjq/akmcgc


## 0. Debug Contract Helpers

下面两个 helper 用来统一打印每一层的输入、输出和参数维度。`parameter contract` 中的每一行都是一个 PyTorch `Parameter`。


In [2]:
def tensor_contract(name, value):
    if torch.is_tensor(value):
        return f"{name:16s} shape={tuple(value.shape)!s:18s} dtype={str(value.dtype):14s} device={value.device}"
    return f"{name:16s} value={value!r}"


def print_tensor_contract(title, tensors):
    print(f"\n[{title}]")
    for name, value in tensors.items():
        print(tensor_contract(name, value))


def print_parameter_contract(title, module, max_rows=30):
    print(f"\n[{title} parameter contract]")
    total = 0
    trainable = 0
    rows = []
    for name, param in module.named_parameters():
        n = param.numel()
        total += n
        if param.requires_grad:
            trainable += n
        rows.append((name, tuple(param.shape), param.dtype, param.requires_grad, n))
    print(f"total parameters={total:,}, trainable={trainable:,}, tensors={len(rows)}")
    for name, shape, dtype, requires_grad, n in rows[:max_rows]:
        print(f"{name:48s} shape={shape!s:18s} dtype={str(dtype):14s} trainable={requires_grad!s:5s} n={n:,}")
    if len(rows) > max_rows:
        print(f"... {len(rows) - max_rows} more parameter tensors")


## 1. Load One H2O Reactant/Product Pair

In [3]:
dataset = ReactionPairDataset(
    react_file=str(data_dir / 'h2o_react.extxyz'),
    product_file=str(data_dir / 'h2o_product.extxyz'),
    cutoff=6.0,
    max_neigh=16,
    r_fixed=True,
    r_pbc=True,
    device='cpu',
)
loader = DataLoader(dataset, batch_size=1, shuffle=False, collate_fn=ReactionPairDataset.collate_fn)
batch = next(iter(loader))

for key in ['h', 'pos', 'edge_index', 'cell_offsets', 'cell', 'pbc', 'fragment', 'mask', 'neighbors']:
    value = batch[key]
    print(f'{key:12s}', tuple(value.shape), value.dtype)

assert len(dataset) == 1
assert batch['h'].dtype == torch.float64
assert torch.allclose(batch['h'][:, :3], batch['pos'])
assert torch.all(batch['fragment'][batch['edge_index'][0]] == batch['fragment'][batch['edge_index'][1]])


h            (6, 122) torch.float64
pos          (6, 3) torch.float64
edge_index   (2, 12) torch.int64
cell_offsets (12, 3) torch.float64
cell         (1, 2, 3, 3) torch.float64
pbc          (1, 2, 3) torch.bool
fragment     (6,) torch.int64
mask         (6,) torch.int64
neighbors    (1,) torch.int64


## 2. Build EGNN And LEFTNet

In [9]:
feature_dim = batch['h'].shape[1] - 3
print(feature_dim)
egnn = EGNN(
    in_node_nf=feature_dim,
    in_edge_nf=0,
    hidden_nf=64,
    edge_hidden_nf=64,
    n_layers=2,
    attention=True,
    out_node_nf=feature_dim,
    tanh=True,
    coords_range=10.0,
    norm_constant=1.0,
    inv_sublayers=2,
    sin_embedding=False,
    normalization_factor=1.0,
    aggregation_method='mean',
    reflect_equiv=True,
).to(dtype=batch['pos'].dtype).eval()

leftnet = LEFTNet(
    pos_require_grad=False,
    cutoff=6.5,
    num_layers=2,
    hidden_channels=64,
    num_radial=32,
    in_hidden_channels=feature_dim,
    reflect_equiv=True,
    legacy=True,
    update=True,
    pos_grad=False,
    single_layer_output=True,
    object_aware=True,
).to(dtype=batch['pos'].dtype).eval()


119


## 2.1 Model Parameter Contracts

这里先看 EGNN / LEFTNet 内部到底有哪些参数矩阵，以及每个矩阵的 shape。


In [10]:
print('feature_dim = batch["h"].shape[1] - 3 =', feature_dim)
print_parameter_contract('EGNN', egnn, max_rows=24)
print_parameter_contract('LEFTNet', leftnet, max_rows=24)


feature_dim = batch["h"].shape[1] - 3 = 119

[EGNN parameter contract]
total parameters=166,971, trainable=166,971, tensors=68
embedding.weight                                 shape=(64, 119)          dtype=torch.float64  trainable=True  n=7,616
embedding.bias                                   shape=(64,)              dtype=torch.float64  trainable=True  n=64
embedding_out.weight                             shape=(119, 64)          dtype=torch.float64  trainable=True  n=7,616
embedding_out.bias                               shape=(119,)             dtype=torch.float64  trainable=True  n=119
edge_embedding.weight                            shape=(63, 1)            dtype=torch.float64  trainable=True  n=63
edge_embedding.bias                              shape=(63,)              dtype=torch.float64  trainable=True  n=63
edge_embedding_out.weight                        shape=(1, 63)            dtype=torch.float64  trainable=True  n=63
edge_embedding_out.bias                          shape

## 3. Forward Equivariance Helpers

In [6]:
def clone_batch(batch):
    return {k: v.clone() if torch.is_tensor(v) else v for k, v in batch.items()}


def run_model(model, batch):
    return model(
        batch['h'][:, 3:],
        batch['pos'],
        batch['edge_index'],
        edge_attr=None,
        cell=batch['cell'],
        pbc=batch['pbc'],
        cell_offsets=batch['cell_offsets'],
        neighbors=batch['neighbors'],
        fragment=batch['fragment'],
        mask=batch['mask'],
    )


def rotate_batch(batch, rot):
    out = clone_batch(batch)
    out['pos'] = out['pos'] @ rot
    out['cell'] = out['cell'] @ rot
    out['h'][:, :3] = out['pos']
    return out


def check_model_equivariance(name, model, batch, atol=2e-5):
    torch.manual_seed(0)
    rot = o3.rand_matrix().to(dtype=batch['pos'].dtype, device=batch['pos'].device)
    batch_rot = rotate_batch(batch, rot)

    with torch.no_grad():
        h, pos, _ = run_model(model, batch)
        h_rot, pos_rot, _ = run_model(model, batch_rot)

    h_err = float((h - h_rot).abs().max())
    pos_err = float((pos @ rot - pos_rot).abs().max())
    print(f'{name} max |h - h_rot| =', h_err)
    print(f'{name} max |pos @ R - pos_rot| =', pos_err)
    assert h_err < atol
    assert pos_err < atol


## 4. Run Checks

## 3.1 Model Forward Input / Output Contract

模型 forward 的输入是 `node_h=batch["h"][:, 3:]` 和 `pos=batch["pos"]`，输出是 `h_out, pos_out, edge_attr_out`。


In [7]:
for name, model in [('EGNN', egnn), ('LEFTNet', leftnet)]:
    print_tensor_contract(f'{name} forward inputs', {
        'node_h': batch['h'][:, 3:],
        'pos': batch['pos'],
        'edge_index': batch['edge_index'],
        'cell': batch['cell'],
        'pbc': batch['pbc'],
        'cell_offsets': batch['cell_offsets'],
        'neighbors': batch['neighbors'],
        'fragment': batch['fragment'],
        'mask': batch['mask'],
    })
    with torch.no_grad():
        h_out, pos_out, edge_attr_out = run_model(model, batch)
    print_tensor_contract(f'{name} forward outputs', {
        'h_out': h_out,
        'pos_out': pos_out,
        'edge_attr_out': edge_attr_out,
    })



[EGNN forward inputs]
node_h           shape=(6, 119)           dtype=torch.float64  device=cpu
pos              shape=(6, 3)             dtype=torch.float64  device=cpu
edge_index       shape=(2, 12)            dtype=torch.int64    device=cpu
cell             shape=(1, 2, 3, 3)       dtype=torch.float64  device=cpu
pbc              shape=(1, 2, 3)          dtype=torch.bool     device=cpu
cell_offsets     shape=(12, 3)            dtype=torch.float64  device=cpu
neighbors        shape=(1,)               dtype=torch.int64    device=cpu
fragment         shape=(6,)               dtype=torch.int64    device=cpu
mask             shape=(6,)               dtype=torch.int64    device=cpu

[EGNN forward outputs]
h_out            shape=(6, 119)           dtype=torch.float64  device=cpu
pos_out          shape=(6, 3)             dtype=torch.float64  device=cpu
edge_attr_out    shape=(12, 1)            dtype=torch.float64  device=cpu

[LEFTNet forward inputs]
node_h           shape=(6, 119)        

In [8]:
check_model_equivariance('EGNN', egnn, batch)
check_model_equivariance('LEFTNet', leftnet, batch)
print('H2O model forward equivariance checks passed.')


EGNN max |h - h_rot| = 1.8880549582100947e-08
EGNN max |pos @ R - pos_rot| = 1.0329703314937433e-08
LEFTNet max |h - h_rot| = 9.666444777955974e-08
LEFTNet max |pos @ R - pos_rot| = 6.966871524127782e-11
H2O model forward equivariance checks passed.
